# Lab 08 · Text & Graph Mining
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 8**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab08_Text_and_Graph_Mining.ipynb)

**In this lab you will:**
- turn text into TF-IDF vectors and inspect important words
- classify sentiment with Naïve Bayes and a linear SVM
- discover topics by clustering documents
- compute centralities, PageRank and communities in a social network

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## Part A · Text mining
### 1 · A small corpus of course reviews (English & Arabic)

In [ ]:
reviews = [
    ("The lectures were clear and the labs were very useful", "pos"),
    ("Excellent course, I learned clustering and classification", "pos"),
    ("The instructor explains difficult concepts with great examples", "pos"),
    ("Very practical labs with real data sets", "pos"),
    ("I enjoyed the projects and the Python notebooks", "pos"),
    ("Great slides and helpful feedback on assignments", "pos"),
    ("The course was well organised and interesting", "pos"),
    ("Useful content for my career in data science", "pos"),
    ("Clear explanations of decision trees and ensembles", "pos"),
    ("I liked the research examples from Saudi Arabia", "pos"),
    ("المحاضرات واضحة والمعامل مفيدة جدا", "pos"),
    ("مقرر ممتاز وتعلمت الكثير", "pos"),
    ("الدكتور يشرح بطريقة رائعة", "pos"),
    ("The lectures were too fast and confusing", "neg"),
    ("Too much theory and not enough practice", "neg"),
    ("The exam was very difficult and unfair", "neg"),
    ("I did not understand the association rules chapter", "neg"),
    ("The labs were boring and the data was messy", "neg"),
    ("Assignments took too much time", "neg"),
    ("Poor organisation and late feedback", "neg"),
    ("The slides had too much text", "neg"),
    ("I was lost during the clustering lectures", "neg"),
    ("The workload was heavy and stressful", "neg"),
    ("المحاضرات سريعة وغير واضحة", "neg"),
    ("الاختبار صعب جدا", "neg"),
    ("الواجبات كثيرة ومرهقة", "neg"),
]
text_df = pd.DataFrame(reviews, columns=["text", "label"])
text_df.sample(5, random_state=1)

### 2 · TF-IDF representation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english")
T = vec.fit_transform(text_df["text"])
print("documents x terms:", T.shape)

# Top words per class (mean TF-IDF weight)
weights = pd.DataFrame(T.toarray(), columns=vec.get_feature_names_out())
top = weights.groupby(text_df["label"]).mean().T
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, lab, col in zip(axes, ["pos", "neg"], ["seagreen", "tomato"]):
    top[lab].nlargest(10).sort_values().plot(kind="barh", ax=ax, color=col, title=f"Top words: {lab}")
plt.tight_layout(); plt.show()

### 3 · Sentiment classification

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, clf in [("Naive Bayes", MultinomialNB()), ("Linear SVM", LinearSVC())]:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), clf)
    print(f"{name:12s} CV accuracy = {cross_val_score(pipe, text_df['text'], text_df['label'], cv=cv).mean():.2f}")

svm = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LinearSVC()).fit(text_df["text"], text_df["label"])
new = ["The labs were really useful", "The exam was confusing", "المعامل مفيدة"]   # ✏️ write your own
print(dict(zip(new, svm.predict(new))))

### 4 · Topic discovery by clustering the documents

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(T)
terms = vec.get_feature_names_out()
for c in range(3):
    centre = km.cluster_centers_[c]
    print(f"Topic {c}:", ", ".join(terms[i] for i in centre.argsort()[::-1][:6]))

## Part B · Graph mining
### 5 · Zachary's karate club network

In [ ]:
import networkx as nx
G = nx.karate_club_graph()
print(G.number_of_nodes(), "members,", G.number_of_edges(), "friendships")
degrees = [d for _, d in G.degree()]
plt.hist(degrees, bins=range(1, max(degrees) + 2), color="teal", edgecolor="white")
plt.xlabel("degree"); plt.ylabel("number of members"); plt.title("Degree distribution"); plt.show()

In [ ]:
cent = pd.DataFrame({
    "degree": nx.degree_centrality(G),
    "betweenness": nx.betweenness_centrality(G),
    "closeness": nx.closeness_centrality(G),
    "pagerank": nx.pagerank(G, alpha=0.85)}).round(3)
cent.sort_values("pagerank", ascending=False).head(8)

In [ ]:
from networkx.algorithms.community import louvain_communities, modularity
communities = louvain_communities(G, seed=0)
colour = {n: i for i, c in enumerate(communities) for n in c}
pos = nx.spring_layout(G, seed=3)
pr = nx.pagerank(G)
plt.figure(figsize=(8, 6))
nx.draw_networkx(G, pos, node_color=[colour[n] for n in G], cmap="Set2",
                 node_size=[4000 * pr[n] for n in G], with_labels=True, font_size=8, edge_color="#bbbbbb")
plt.title(f"{len(communities)} Louvain communities, modularity = {modularity(G, communities):.2f}  (size = PageRank)")
plt.axis("off"); plt.show()

## ✏️ Exercises
1. Add 10 new reviews of your own and retrain. Does accuracy change?
2. Use `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))` — useful for Arabic morphology.
3. Which member has the highest betweenness? Why might that person be a 'broker'?
4. Compare the Louvain communities with the real split stored in `G.nodes[n]["club"]`.